# Segregating ESA land cover map tree into three: needle leaf, broad leaf and mixed forest

In [1]:
# Function to segregate the ESA land cover map tree by the North American Environmental Atlas - Land Cover 2020 30m tree class
import xarray as xr
import numpy as np
import pandas as pd


def update_mesh_gru_database(
    landsat_path: str,
    esa_path: str,
    output_path: str,
    gru_name_list=None,
    save=True
):
    """
    Updates the GRU fractions in a MESH drainage database NetCDF file using
    proportions from another GRU dataset (ESA → Landsat logic).

    Parameters
    ----------
    landsat_path : str
        Path to the Landsat-based MESH_drainage_database.nc file.
    esa_path : str
        Path to the ESA-based MESH_drainage_database.nc file.
    output_path : str
        Output file path for the revised NetCDF.
    gru_name_list : list[str], optional
        Custom list of GRU names to assign (default provided).
    save : bool
        Whether to save the output NetCDF file.

    Returns
    -------
    xr.Dataset
        Updated dataset with modified GRU fractions.
    """

    # ---- Load Data ----
    db_landsat = xr.open_dataset(landsat_path)
    db_esa = xr.open_dataset(esa_path)

    # Convert to DataFrames (easier for column-based logic)
    landsat_gru = pd.DataFrame(db_landsat["GRU"].values)
    landsat_gru.columns = [f"frac_{i}" for i in landsat_gru.columns]

    esa_gru = pd.DataFrame(db_esa["GRU"].values)
    esa_gru.columns = [f"frac_{i}" for i in esa_gru.columns]

    # ---- Step A: Adjust Landsat fractions ----
    # Move frac_1 → frac_2
    landsat_gru["frac_2"] += landsat_gru["frac_1"]

    # Columns to proportionally redistribute
    redistribute_cols = ["frac_2", "frac_3", "frac_4"]

    col_sum = landsat_gru[redistribute_cols].sum(axis=1)

    # Compute proportions safely
    proportions = landsat_gru[redistribute_cols].div(
        col_sum.replace(0, np.nan), axis=0
    )

    # Zero-sum rows → force frac_2 = 1
    zero_mask = col_sum == 0
    proportions.loc[zero_mask] = [1, 0, 0]

    # Multiply by ESA frac_0
    result = proportions.mul(esa_gru["frac_0"], axis=0)
    result.columns = redistribute_cols

    # ---- Step B: Reconstruct new GRU matrix ----
    landsat_gru.loc[:, :] = 0  # Reset all

    # Assign redistributed fractions
    landsat_gru["frac_1"] = result["frac_2"]  # Note mapping from old logic
    landsat_gru["frac_3"] = result["frac_3"]
    landsat_gru["frac_4"] = result["frac_4"]

    # Copy additional GRUs from ESA, according to provided mapping
    mapping = {
        "frac_5": "frac_1",
        "frac_6": "frac_2",
        "frac_8": "frac_9",
        "frac_10": "frac_8",
        "frac_11": "frac_3",
        "frac_12": "frac_5",
        "frac_13": "frac_4",
        "frac_14": "frac_7",
        "frac_15": "frac_6",
    }

    for dst, src in mapping.items():
        landsat_gru[dst] = esa_gru[src]

    # ---- Step C: Convert back to dataset ----
    new_gru_matrix = landsat_gru.to_numpy()

    # Replace GRU variable
    gru_attrs = db_landsat["GRU"].attrs.copy()
    db_landsat["GRU"] = (["subbasin", "NGRU"], new_gru_matrix)
    db_landsat["GRU"].attrs = gru_attrs

    # GRU name list
    if gru_name_list is None:
        gru_name_list = [
            'Unknown', 'Needleleaf forest', 'Sub-polar taiga needleleaf forest',
            'Broadleaf forest', 'Mixed forest', 'Shrubland', 'Grassland',
            'Sub-polar or polar shrubland-lichen-moss', 'Moss and lichen',
            'Sub-polar or polar barren-lichen-moss', 'Herbaceous wetland',
            'Cropland', 'Bare-sparse vegetation', 'Built-up',
            'Permanent water bodies', 'Snow and ice', 'Dump'
        ]

    db_landsat["LandUse"] = (["NGRU"], np.array(gru_name_list, dtype=object))
    db_landsat["LandUse"].attrs = {
        "standard_name": "Landuse classification name",
        "units": "dimensionless"
    }

    # ---- Save output ----
    if save:
        db_landsat.to_netcdf(output_path)

    return

# Call the function with inputs
update_mesh_gru_database(
    landsat_path="/home/zelalem/mesh_outputs_landsat/MESH_drainage_database.nc",
    esa_path="/home/zelalem/mesh_outputs_esa/MESH_drainage_database.nc",
    output_path="/home/zelalem/mesh_outputs_esa/MESH_drainage_database_rev.nc"
)   

# Polish GRUs (landcover) fraction based on a give threshold values [0.1%, 1%, 2%, 5%]

A function designed to combine or consolidate GRU fraction values by applying specified minimum threshold criteria. Specifically, any individual GRU fraction that falls below these defined threshold values will regrouped and summed into dominant GRUs, while fractions meeting or exceeding the threshold remain separate. It's also better to use this script after the "3-specific" step's outputs.

In [2]:
# GRUS aggregation function: polish small GRUs fractions for computation efficiency 
# load the required library
import pandas as pd
import numpy as np

# define the function
def aggregate_grus_fraction(input_grus, nonveg_grus, glacier_grus, key_id, startcolname, 
                           minimum_gl_fraction, minimum_nonveg_fraction, minimum_veg_fraction):
    """
    Normalize and adjust GRU fractions based on category thresholds for computational efficiency.
    
    Parameters:
    - input_grus (pd.DataFrame): DataFrame with GRU fraction columns.
    - nonveg_grus (list): List of non-vegetated GRU column names.
    - glacier_grus (list): List of glacier GRU column names.
    - key_id (str): Column name for sorting.
    - startcolname (str): Prefix for GRU columns.
    - minimum_gl_fraction (float): Minimum fraction threshold for glacier GRUs.
    - minimum_nonveg_fraction (float): Minimum fraction threshold for nonveg GRUs.
    - minimum_veg_fraction (float): Minimum fraction threshold for veg GRUs.
    
    Returns:
    - pd.DataFrame: Adjusted DataFrame with normalized GRU fractions.
    """
    # Validate inputs
    if not isinstance(input_grus, pd.DataFrame):
        raise ValueError("input_grus must be a pandas DataFrame")
    if key_id not in input_grus.columns:
        print(f"key_id '{key_id}' not found in input_grus columns")
    if not all(isinstance(x, (int, float)) and x >= 0 for x in 
               [minimum_gl_fraction, minimum_nonveg_fraction, minimum_veg_fraction]):
        raise ValueError("Minimum fractions must be non-negative numbers")
        
   # Sort by key_id
    if key_id in input_grus.columns:
        input_grus = input_grus.sort_values(by=key_id).reset_index(drop=True)
    
    # Identify all GRU columns
    all_grus = [col for col in input_grus.columns if col.startswith(startcolname)]
    if not all_grus:
        raise ValueError(f"No columns found starting with '{startcolname}'")
    
    # Ensure mutually exclusive categories
    glacier_grus = [col for col in all_grus if col in (glacier_grus or [])]
    nonveg_grus = [col for col in all_grus if col in (nonveg_grus or []) and col not in glacier_grus]
    veg_grus = [col for col in all_grus if col not in (glacier_grus + nonveg_grus)]
    
    # Validate category columns
    invalid_cols = set(glacier_grus + nonveg_grus) - set(all_grus)
    if invalid_cols:
        raise ValueError(f"Invalid GRU columns: {invalid_cols}")
    
    # Normalize fractions (row sums = 1)
    row_sums = input_grus[all_grus].sum(axis=1).replace(0, 1)
    input_grus[all_grus] = input_grus[all_grus].div(row_sums, axis=0)
    
    # Calculate original category sums
    for grus, label in zip([glacier_grus, nonveg_grus, veg_grus], ['gl', 'nonveg', 'veg']):
        if grus:
            input_grus[f'{label}_grus_sum'] = input_grus[grus].sum(axis=1)
        else:
            input_grus[f'{label}_grus_sum'] = 0.0
    
    # Apply minimum fraction thresholds
    for col in all_grus:
        threshold = (minimum_gl_fraction if col in glacier_grus else 
                     minimum_nonveg_fraction if col in nonveg_grus else 
                     minimum_veg_fraction)
        input_grus[col] = input_grus[col].where(input_grus[col] >= threshold, 0)
    
    # Calculate new category sums after thresholding
    for grus, label in zip([glacier_grus, nonveg_grus, veg_grus], ['gl', 'nonveg', 'veg']):
        if grus:
            input_grus[f'{label}_grus_sum_new'] = input_grus[grus].sum(axis=1)
        else:
            input_grus[f'{label}_grus_sum_new'] = 0.0
    
    input_grus['all_grus_sum_new'] = input_grus[all_grus].sum(axis=1)
    
    # Renormalize fractions to preserve original category sums
    for grus, label in zip([glacier_grus, nonveg_grus, veg_grus], ['gl', 'nonveg', 'veg']):
        if grus:
            # Compute scaling factor: original_sum / new_sum
            scale = input_grus[f'{label}_grus_sum'] / input_grus[f'{label}_grus_sum_new']
            scale = scale.replace([np.inf, -np.inf], 0).fillna(1)
            # Apply scaling to individual GRUs
            input_grus[grus] = input_grus[grus].mul(scale, axis=0)
    
    # Final normalization to ensure row sums = 1
    row_sums_new = input_grus[all_grus].sum(axis=1).replace(0, 1)
    input_grus[all_grus] = input_grus[all_grus].div(row_sums_new, axis=0)
    
    # Drop intermediate columns
    cols_to_drop = [f'{label}_grus_sum' for label in ['gl', 'nonveg', 'veg']] + \
                   [f'{label}_grus_sum_new' for label in ['gl', 'nonveg', 'veg']] + \
                   ['all_grus_sum_new']
    input_grus.drop(columns=[col for col in cols_to_drop if col in input_grus.columns], 
                    inplace=True)
    return input_grus


# Function to calculate the number of positive fraction GRUs to evaluate the level of polishing 
import pandas as pd
def count_positive_fractions(df, prefix='frac_'):
    # Identify columns starting with the prefix
    frac_cols = [col for col in df.columns if col.startswith(prefix)]
    if not frac_cols:
        raise ValueError(f"No columns found starting with '{prefix}'")
    # Count values > 0 for each matching column
    counts = df[frac_cols].gt(0).sum()
    # Add total count of all positive values in these columns
    total_count = df[frac_cols].gt(0).sum().sum()
    counts['total'] = total_count
    return counts

In [3]:
## # Example use
# glacier_grus = ['frac_19']                                   # List of GRUs (Glacierd)
# nonveg_grus = ['frac_16', 'frac_17', 'frac_18', 'frac_19']   # List of GRUs (Glacier, Water, Urban, Barrenland)
# minimum_gl_fraction=0.01
# minimum_veg_fraction=0.05 
# minimum_nonveg_fraction=0.01
# startcolname = 'frac_'
# key_id = 'ID_s'
# input_grus = aggregate_grus_fraction(input_grus, nonveg_grus, glacier_grus, key_id, startcolname, minimum_gl_fraction, minimum_nonveg_fraction, minimum_veg_fraction)

# Example use:
# Count positive values in 'frac_' columns
# result = count_positive_fractions(df)
# print("Count of values > 0 in each 'frac_' column:")
# print(result)

We use the `Polish GRUs` Python script to build a `MESH` model setup for the Canada and Transboundary River Basin.

In [5]:
# Import the required library: 'netCDF', 'shutil', 'xarray' to read and re-save the NetCDF file.
import xarray as xr
import netCDF4 as nc
import shutil

# Path to the original NetCDF file
ddbnetcdf_path = '/home/zelalem/mesh_outputs_esa/MESH_drainage_database_rev.nc'

# Path where the modified NetCDF file will be saved 
ddbnetcdf_pathsave = '/home/zelalem/mesh_outputs_esa/MESH_drainage_database_Polish_0p05_0p02_0p01.nc'

# Open the dataset
db = xr.open_dataset(ddbnetcdf_path)

# Copy the attribute information for later use
gru_attrs = db['GRU'].attrs.copy()
landuse_attrs = db['LandUse'].attrs.copy()

# Read and save the variables that will be replaced to local variables.
gru_frac = pd.DataFrame(db['GRU'].values)

# Add "frac_" prefix to each column name
gru_frac.columns = ['frac_' + str(col) for col in gru_frac.columns]

# Read and save the variables that will be replaced to local variables.
gru_names = db['LandUse'].values.tolist()

# Drop the fields that will be replaced from the dataset.
db = db.drop_vars(['GRU', 'LandUse'])

# Save the size of the dimensions to local variables.
nsubbasin = db.sizes['subbasin']

print(gru_frac.columns)
print(gru_names)

Index(['frac_0', 'frac_1', 'frac_2', 'frac_3', 'frac_4', 'frac_5', 'frac_6',
       'frac_7', 'frac_8', 'frac_9', 'frac_10', 'frac_11', 'frac_12',
       'frac_13', 'frac_14', 'frac_15', 'frac_16'],
      dtype='object')
['Unknown', 'Needleleaf forest', 'Sub-polar taiga needleleaf forest', 'Broadleaf forest', 'Mixed forest', 'Shrubland', 'Grassland', 'Sub-polar or polar shrubland-lichen-moss', 'Moss and lichen', 'Sub-polar or polar barren-lichen-moss', 'Herbaceous wetland', 'Cropland', 'Bare-sparse vegetation', 'Built-up', 'Permanent water bodies', 'Snow and ice', 'Dump']


In [6]:
## GRU polishing to reduce computation for a given level of GRUs fraction
# Specify the [GlacierGRUs, NonVegGRUs, RemoveGRUs] columns
glacier_grus = ['frac_15']         # List of GRUs (Glacier)
nonveg_grus = ['frac_10', 'frac_12', 'frac_13', 'frac_14', 'frac_16']   # List of GRUs (Wetland, Barrenlands, Urban, Water)
gru_to_remove = ['frac_0']

# Specify the minimum threshold for [VegGRUs, NonVegGRUs, GlacierGRUs]
minimum_veg_fraction=0.05
minimum_nonveg_fraction=0.02
minimum_gl_fraction=0.01
startcolname = 'frac_'
key_id = 'ID_s'

# Excludes the GRU column that listed as Unknown or add them to the group they belong to:
# Comparision between high resolution ESA-10m land cover suggest that 
# most of frac_0 are water so we added them to water GRUs
gru_frac['frac_14'] += gru_frac['frac_0']

# Remove the unwanted or merged GRUs from both the gru_names and gru_fraction
index_gru_to_remove = [gru_frac.columns.get_loc(col) for col in gru_to_remove]
for idx in sorted(index_gru_to_remove, reverse=True):
    del gru_names[idx]
gru_frac = gru_frac.drop(columns = gru_to_remove)

# call for aggregation function 
input_grus = aggregate_grus_fraction(gru_frac, nonveg_grus, glacier_grus, key_id, startcolname, minimum_gl_fraction, minimum_nonveg_fraction, minimum_veg_fraction)

# sanity check
row_sums = input_grus.sum(axis=1)
print("Row sum sanity check — min:", np.nanmin(row_sums), ", max:", np.nanmax(row_sums))

# calculate the reduction of tiles
result = count_positive_fractions(input_grus)
print("Count of values > 0 in each 'frac_' column:")
print(result)

key_id 'ID_s' not found in input_grus columns
Row sum sanity check — min: 0.0 , max: 1.0000000000000004
Count of values > 0 in each 'frac_' column:
frac_1      49157
frac_2          0
frac_3      13712
frac_4      19754
frac_5      11920
frac_6      59053
frac_7          0
frac_8      25332
frac_9          0
frac_10     10365
frac_11     10624
frac_12     16742
frac_13      2515
frac_14     46566
frac_15      6039
frac_16         0
total      271779
dtype: int64


In [ ]:
## level of polishing and number of tiles 
# Case (1) = [0.01, 0.01, 0.01]; Number of tiles = 358798
# Case (2) = [0.02, 0.01, 0.01]; Number of tiles = 334408 (7% reduction)
Case (3) = [0.05, 0.02, 0.01]; Number of tiles = 276031 (23% reduction)

In [7]:
# Convert pandas DataFrame back to NumPy array
new_gru_frac = input_grus.to_numpy()

# Update the revised GRU count.
NGRU = len(input_grus.columns)

# Save new names for the GRUs.
new_gru_names = np.array(gru_names, dtype=object)
 
# Define a dimension for the GRUs in the dataset.
# db.coords['NGRU'] = np.arange(1, (NGRU + 1), dtype = 'int32') 
    
# Save the updated fields to the dataset.
db['GRU'] = (['subbasin', 'NGRU'], new_gru_frac)
db['GRU'].attrs = gru_attrs
db['LandUse'] = (['NGRU'], new_gru_names)
db['LandUse'].attrs = landuse_attrs

# Save the modified file.
db.to_netcdf(ddbnetcdf_pathsave)

____